# Forecasting de ventas
Este notebook se utilizar? para generar predicciones de ventas a partir de nuevos datos de inferencia. El objetivo es replicar las transformaciones del notebook de entrenamiento, dejar preparado `inferencia_df` con la misma estructura de entrada del modelo final y conservar ?nicamente los registros de noviembre para la inferencia.


In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from pathlib import Path
import holidays
import joblib


In [23]:
ruta_inferencia = Path('../data/raw/inferencia/ventas_2025_inferencia.csv')
inferencia_2025 = pd.read_csv(ruta_inferencia)

inferencia_2025.head()


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


In [24]:
ruta_modelo = Path('models/modelofinal.joblib')
modelo_final = joblib.load(ruta_modelo)

print('Modelo final cargado correctamente.')
print(f'Numero de variables esperadas por el modelo: {len(modelo_final.feature_names_in_)}')


Modelo final cargado correctamente.
Numero de variables esperadas por el modelo: 440


In [25]:
# Objetivo: aplicar a inferencia_df las mismas transformaciones del notebook de entrenamiento, adaptadas a inferencia.
inferencia_df = inferencia_2025.copy()

# Si el modelo final no esta cargado en memoria, lo cargamos automaticamente.
if 'modelo_final' not in globals():
    ruta_modelo = Path('models/modelofinal.joblib')
    modelo_final = joblib.load(ruta_modelo)

# Limpieza basica y tipos de datos.
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'], errors='coerce')
inferencia_df = inferencia_df.drop_duplicates().copy()

# En inferencia no eliminamos filas por nulos en ventas observadas.
# Solo descartamos filas si faltan campos indispensables para construir las features del modelo.
columnas_indispensables = ['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'precio_venta', 'es_estrella', 'Amazon', 'Decathlon', 'Deporvillage']
inferencia_df = inferencia_df.dropna(subset=columnas_indispensables).copy()

# Variables temporales principales.
inferencia_df['anio'] = inferencia_df['fecha'].dt.year
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['nombre_mes'] = inferencia_df['fecha'].dt.month_name()
inferencia_df['dia'] = inferencia_df['fecha'].dt.day
inferencia_df['dia_anio'] = inferencia_df['fecha'].dt.dayofyear
inferencia_df['semana_anio'] = inferencia_df['fecha'].dt.isocalendar().week.astype(int)
inferencia_df['dia_semana_num'] = inferencia_df['fecha'].dt.weekday
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.day_name()
inferencia_df['es_fin_de_semana'] = inferencia_df['dia_semana_num'].isin([5, 6])
inferencia_df['es_inicio_mes'] = inferencia_df['fecha'].dt.is_month_start
inferencia_df['es_fin_mes'] = inferencia_df['fecha'].dt.is_month_end
inferencia_df['es_inicio_trimestre'] = inferencia_df['fecha'].dt.is_quarter_start
inferencia_df['es_fin_trimestre'] = inferencia_df['fecha'].dt.is_quarter_end
inferencia_df['es_inicio_anio'] = inferencia_df['fecha'].dt.is_year_start
inferencia_df['es_fin_anio'] = inferencia_df['fecha'].dt.is_year_end
inferencia_df['es_primera_quincena'] = inferencia_df['dia'] <= 15
inferencia_df['es_ultima_quincena'] = inferencia_df['dia'] > 15
inferencia_df['es_payday_inicio_mes'] = inferencia_df['dia'].isin([1, 2, 3, 4, 5])
inferencia_df['es_payday_fin_mes'] = inferencia_df['dia'] >= 25

# Festivos en Espa?a y variables derivadas.
festivos_es = holidays.ES(years=sorted(inferencia_df['anio'].dropna().unique().tolist()))
inferencia_df['es_festivo'] = inferencia_df['fecha'].dt.date.astype('object').isin(festivos_es)
inferencia_df['nombre_festivo'] = inferencia_df['fecha'].dt.date.map(lambda x: festivos_es.get(x))
inferencia_df['es_vispera_festivo'] = (inferencia_df['fecha'] + pd.Timedelta(days=1)).dt.date.astype('object').isin(festivos_es)
inferencia_df['es_post_festivo'] = (inferencia_df['fecha'] - pd.Timedelta(days=1)).dt.date.astype('object').isin(festivos_es)

def black_friday(fecha):
    noviembre = pd.date_range(start=f'{fecha.year}-11-01', end=f'{fecha.year}-11-30', freq='D')
    viernes = noviembre[noviembre.weekday == 4]
    return fecha.normalize() == viernes[-1] if len(viernes) > 0 else False

def cyber_monday(fecha):
    noviembre = pd.date_range(start=f'{fecha.year}-11-01', end=f'{fecha.year}-11-30', freq='D')
    viernes = noviembre[noviembre.weekday == 4]
    return fecha.normalize() == (viernes[-1] + pd.Timedelta(days=3)) if len(viernes) > 0 else False

inferencia_df['es_black_friday'] = inferencia_df['fecha'].apply(black_friday)
inferencia_df['es_cyber_monday'] = inferencia_df['fecha'].apply(cyber_monday)
inferencia_df['dias_hasta_fin_mes'] = inferencia_df['fecha'].dt.days_in_month - inferencia_df['dia']
inferencia_df['semana_mes'] = ((inferencia_df['dia'] - 1) // 7) + 1
inferencia_df['temporada_rebajas_invierno'] = inferencia_df['mes'].isin([1, 2])
inferencia_df['temporada_rebajas_verano'] = inferencia_df['mes'].isin([7, 8])
inferencia_df['campana_navidad'] = inferencia_df['mes'].isin([11, 12])
inferencia_df['campana_vuelta_al_cole'] = inferencia_df['mes'].isin([8, 9])
inferencia_df['es_puente'] = (~inferencia_df['es_festivo']) & (inferencia_df['es_vispera_festivo'] | inferencia_df['es_post_festivo'])

# Lags y media movil: se construyen por producto y por a?o sin eliminar noviembre por nulos en ventas.
# En inferencia los nulos en ventas observadas de noviembre son esperables porque son los registros a predecir.
inferencia_df = inferencia_df.sort_values(['producto_id', 'anio', 'fecha']).copy()
for i in range(1, 8):
    inferencia_df[f'lag_{i}'] = inferencia_df.groupby(['producto_id', 'anio'])['unidades_vendidas'].shift(i)

inferencia_df['media_movil_7_dias'] = (
    inferencia_df.groupby(['producto_id', 'anio'])['unidades_vendidas']
    .transform(lambda serie: serie.shift(1).rolling(window=7, min_periods=7).mean())
)

# Variables de precio y competencia.
inferencia_df['descuento_porcentaje'] = ((inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base']) * 100
columnas_competencia = ['Amazon', 'Decathlon', 'Deporvillage']
inferencia_df['precio_competencia'] = inferencia_df[columnas_competencia].mean(axis=1)
inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']
inferencia_df = inferencia_df.drop(columns=columnas_competencia)

# Copias y one hot encoding de variables categoricas.
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']
columnas_ohe = ['nombre_h', 'categoria_h', 'subcategoria_h']
inferencia_df = pd.get_dummies(inferencia_df, columns=columnas_ohe, dtype=int)

# Dejamos solo las columnas esperadas por el modelo final.
features_modelo_final = list(modelo_final.feature_names_in_)
for col in features_modelo_final:
    if col not in inferencia_df.columns:
        inferencia_df[col] = 0

inferencia_df_modelo = inferencia_df[features_modelo_final].copy()

# Conservamos solo noviembre y eliminamos octubre sin borrar filas por ausencia de ventas observadas.
inferencia_df = inferencia_df[inferencia_df['mes'] == 11].copy()
inferencia_df_modelo = inferencia_df_modelo.loc[inferencia_df.index].copy()

print(f'Forma final de inferencia_df: {inferencia_df.shape}')
print('Columnas finales de inferencia_df:')
print(inferencia_df.columns.tolist())
print(f'Numero de variables de entrada del modelo: {len(inferencia_df_modelo.columns)}')
print('Nulos en variables de entrada del modelo para noviembre:')
display(inferencia_df_modelo.isna().sum()[inferencia_df_modelo.isna().sum() > 0].sort_values(ascending=False))
display(inferencia_df[['fecha', 'producto_id', 'nombre', 'unidades_vendidas']].head())


Forma final de inferencia_df: (720, 98)
Columnas finales de inferencia_df:
['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta', 'ingresos', 'anio', 'trimestre', 'mes', 'nombre_mes', 'dia', 'dia_anio', 'semana_anio', 'dia_semana_num', 'dia_semana', 'es_fin_de_semana', 'es_inicio_mes', 'es_fin_mes', 'es_inicio_trimestre', 'es_fin_trimestre', 'es_inicio_anio', 'es_fin_anio', 'es_primera_quincena', 'es_ultima_quincena', 'es_payday_inicio_mes', 'es_payday_fin_mes', 'es_festivo', 'nombre_festivo', 'es_vispera_festivo', 'es_post_festivo', 'es_black_friday', 'es_cyber_monday', 'dias_hasta_fin_mes', 'semana_mes', 'temporada_rebajas_invierno', 'temporada_rebajas_verano', 'campana_navidad', 'campana_vuelta_al_cole', 'es_puente', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'media_movil_7_dias', 'descuento_porcentaje', 'precio_competencia', 'ratio_precio', 'nombre_h_Adidas Own The Run Jacket', 'nombre_h

lag_1                 696
media_movil_7_dias    696
lag_2                 672
lag_3                 648
lag_4                 624
lag_5                 600
lag_6                 576
lag_7                 552
dtype: int64

,fecha,producto_id,nombre,unidades_vendidas
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,NaN
192,2025-11-02,PROD_001,Nike Air Zoom Pegasus 40,NaN
216,2025-11-03,PROD_001,Nike Air Zoom Pegasus 40,NaN
240,2025-11-04,PROD_001,Nike Air Zoom Pegasus 40,NaN
264,2025-11-05,PROD_001,Nike Air Zoom Pegasus 40,NaN


In [26]:
ruta_salida = Path('../data/processed/inferencia_df_transformado.csv')
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
inferencia_df.to_csv(ruta_salida, index=False)

print(f'DataFrame transformado guardado en: {ruta_salida}')


DataFrame transformado guardado en: ..\data\processed\inferencia_df_transformado.csv


In [27]:
inferencia_df.shape

(720, 98)